# FireSight-IR | Module 1a — Download VIIRS & FIRMS Data

**Project:** FireSight-IR — FireSat Protoflight-aligned wildfire detection pipeline  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Module:** 1a of 9 — Data ingestion: VIIRS VNP14IMG + FIRMS active fire ground truth  
**Platform:** Google Colab (cloud-native, no local downloads)  
**Storage:** Google Drive at `/content/drive/MyDrive/firesight-ir/`

---

## What this notebook does

1. Mounts Google Drive for persistent storage across sessions
2. Installs and verifies all required Python packages
3. Authenticates with NASA Earthdata via `earthaccess`
4. Downloads **FIRMS VIIRS S-NPP** annual fire detection CSVs (2018–2023) via the FIRMS API, filtered to the western CONUS domain and fire season months
5. Streams **VNP14IMG** (375 m active fire product) granules directly from NASA Earthdata Cloud — no files saved locally
6. Extracts fire pixel variables from each granule into a structured DataFrame
7. Saves processed parquet files to Google Drive
8. Produces a spatial summary figure

## Domain and date range

| Parameter | Value | Rationale |
|---|---|---|
| Region | Western CONUS | High fire frequency, gas flare false-positive source, diverse land cover |
| Bounding box | lon −125° to −109°, lat 32° to 49° | CA, OR, NV, AZ, ID, MT, WA |
| Training years | 2018–2022 | 5 full fire seasons including 2020 Creek Fire and 2021 Dixie Fire |
| Validation year | 2023 | Held out — never seen during training |
| Fire season | May–October | Months 5–10 |

## Output files

```
Google Drive: firesight-ir/
  data/
    raw/
      firms/        ← FIRMS parquet files (one per year)
    processed/
      viirs_fp/     ← VNP14IMG fire pixel parquet files (one per year)
  figures/          ← spatial summary plots
```

---
## Section 0 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted at /content/drive')

ModuleNotFoundError: No module named 'google.colab'

---
## Section 1 — Install and verify packages

Run this cell once per Colab session. Colab resets its environment on reconnect so packages must be reinstalled each time. Google Drive files persist.

In [ ]:
# Install all required packages
# earthaccess  — NASA Earthdata authentication and cloud data access
# requests     — HTTP client for FIRMS API calls
# pyarrow      — required backend for parquet file I/O
# h5netcdf     — HDF5/NetCDF reader used by xarray for VNP14IMG files
!pip install earthaccess requests pandas numpy xarray h5py h5netcdf pyarrow tqdm -q

# Verify all imports
import earthaccess
import requests
import pandas as pd
import numpy as np
import xarray as xr
import h5py
import os
import time
import warnings
from io import StringIO
from datetime import date, timedelta
from pathlib import Path

print('Package versions:')
print(f'  earthaccess : {earthaccess.__version__}')
print(f'  pandas      : {pd.__version__}')
print(f'  numpy       : {np.__version__}')
print(f'  xarray      : {xr.__version__}')
print(f'  h5py        : {h5py.__version__}')
print('All imports OK')

---
## Section 2 — Project configuration

All configuration lives in one cell. Edit values here — do not hardcode them elsewhere.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/firesight-ir'
FIRMS_DIR   = f'{BASE_DIR}/data/raw/firms'
VIIRS_DIR   = f'{BASE_DIR}/data/processed/viirs_fp'
FIGURES_DIR = f'{BASE_DIR}/figures'

for d in [FIRMS_DIR, VIIRS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Spatial domain — western CONUS ────────────────────────────────────────────
# earthaccess uses tuple: (west, south, east, north)
BBOX_TUPLE = (-125, 32, -109, 49)
# FIRMS API uses string: 'west,south,east,north'
BBOX_STR   = '-125,32,-109,49'
# Dict form for spatial filtering DataFrames
BBOX_DICT  = {'lon_min': -125.0, 'lon_max': -109.0, 'lat_min': 32.0, 'lat_max': 49.0}

# ── Temporal domain ───────────────────────────────────────────────────────────
TRAIN_YEARS         = [2018, 2019, 2020, 2021, 2022]
VAL_YEAR            = 2023
ALL_YEARS           = TRAIN_YEARS + [VAL_YEAR]
FIRE_SEASON_MONTHS  = [5, 6, 7, 8, 9, 10]   # May–October

# ── FIRMS configuration ───────────────────────────────────────────────────────
# Get your free map key at: https://firms.modaps.eosdis.nasa.gov/api/map_key/
# Log in with your NASA Earthdata credentials to see your key
FIRMS_MAP_KEY   = '37ac8057666042849871ea1b5de95abb'   # replace with your own key
FIRMS_SOURCE    = 'VIIRS_SNPP_SP'                       # SP = Standard Processing (historical archive)

# ── VIIRS product ─────────────────────────────────────────────────────────────
VIIRS_SHORT_NAME = 'VNP14IMG'    # 375 m active fire, Suomi NPP, Collection 2
MIN_FIRMS_COUNT  = 10            # minimum FIRMS detections on a day to query VIIRS

print('Configuration set.')
print(f'  Training years : {TRAIN_YEARS}')
print(f'  Validation year: {VAL_YEAR}')
print(f'  Domain         : lon {BBOX_DICT["lon_min"]}–{BBOX_DICT["lon_max"]}, lat {BBOX_DICT["lat_min"]}–{BBOX_DICT["lat_max"]}')
print(f'  Fire season    : months {FIRE_SEASON_MONTHS}')
print(f'  Storage root   : {BASE_DIR}')

---
## Section 3 — NASA Earthdata authentication

`earthaccess` handles NASA Earthdata authentication and provides direct streaming access to NASA cloud data (S3). You need a free NASA Earthdata account.

**Register here:** https://urs.earthdata.nasa.gov/users/new

In [ ]:
# Interactive login — prompts for username and password
# Credentials are NOT stored in the notebook
auth = earthaccess.login(strategy='interactive')
print(f'Authenticated: {auth.authenticated}')

---
## Section 4 — Download FIRMS active fire ground truth

**FIRMS** (Fire Information for Resource Management System) provides near-real-time and historical satellite fire detections. We use **VIIRS S-NPP Standard Processing (SP)** — the quality-controlled archive product suitable for scientific analysis.

### Why FIRMS is the ground truth

FIRMS active fire detections are the labels that supervise all model training in FireSight-IR. Each detection is a confirmed thermal anomaly at 375 m resolution with associated brightness temperatures, fire radiative power (FRP), and confidence scores. By filtering to `type=0` (presumed vegetation fire) we get clean wildfire labels. Non-fire thermal emitters (`type=2, 3`) become negative training examples for the false-alarm discriminator in Module 3.

### FIRMS API structure

The API endpoint is:
```
https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{SOURCE}/{BBOX}/{DAY_RANGE}/{DATE}
```
- `DAY_RANGE` is capped at 5 days per request
- For full years we chunk into 5-day blocks and concatenate
- Rate limit: 5,000 requests per 10 minutes

In [ ]:
def fetch_firms_year(year, key, bbox_str, fire_months, out_dir):
    """
    Fetch one full year of FIRMS VIIRS fire detections by chunking
    the year into 5-day API requests and concatenating results.

    Parameters
    ----------
    year       : int   — calendar year to fetch
    key        : str   — FIRMS map key
    bbox_str   : str   — bounding box as 'west,south,east,north'
    fire_months: list  — months to keep (e.g. [5,6,7,8,9,10])
    out_dir    : str   — directory to save parquet file

    Returns
    -------
    pd.DataFrame or None
    """
    out_path = f'{out_dir}/firms_viirs_snpp_{year}.parquet'
    if os.path.exists(out_path):
        print(f'  {year} already exists — loading from Drive')
        return pd.read_parquet(out_path)

    all_chunks = []
    current    = date(year, 1, 1)
    end        = date(year, 12, 31)
    n_chunks   = ((end - current).days // 5) + 1
    print(f'  Fetching {year} in {n_chunks} chunks of 5 days...')

    while current <= end:
        date_str = current.strftime('%Y-%m-%d')
        url = (
            f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/'
            f'{key}/{FIRMS_SOURCE}/{bbox_str}/5/{date_str}'
        )
        try:
            resp = requests.get(url, timeout=30)
            if resp.status_code == 200 and len(resp.text) > 100:
                chunk = pd.read_csv(StringIO(resp.text))
                if len(chunk) > 0:
                    all_chunks.append(chunk)
        except Exception as e:
            pass   # transient errors — silently skip and continue
        current += timedelta(days=5)
        time.sleep(0.2)   # respect FIRMS rate limit

    if not all_chunks:
        print(f'  No data returned for {year}')
        return None

    # Combine and clean
    df = pd.concat(all_chunks, ignore_index=True).drop_duplicates()
    df['acq_date'] = pd.to_datetime(df['acq_date'])
    df['month']    = df['acq_date'].dt.month

    # Fire season filter
    df = df[df['month'].isin(fire_months)].copy()

    df.to_parquet(out_path, index=False)
    print(f'  {year}: {len(df):,} detections saved → Drive')
    return df


print('FIRMS download function defined.')

In [ ]:
# ── Run FIRMS downloads for all years ─────────────────────────────────────────
firms_data = {}

for year in ALL_YEARS:
    print(f'\n── {year} ──')
    df = fetch_firms_year(
        year        = year,
        key         = FIRMS_MAP_KEY,
        bbox_str    = BBOX_STR,
        fire_months = FIRE_SEASON_MONTHS,
        out_dir     = FIRMS_DIR,
    )
    if df is not None:
        firms_data[year] = df

# Combined dataset
firms_all = pd.concat(firms_data.values(), ignore_index=True)
firms_all['acq_date'] = pd.to_datetime(firms_all['acq_date'])

print('\n' + '='*45)
print('  FIRMS Summary')
print('='*45)
total = 0
for year, df in firms_data.items():
    print(f'  {year}: {len(df):>9,} detections')
    total += len(df)
print(f'  {"TOTAL":}: {total:>9,} detections')
print('='*45)
print(f'\nColumns: {firms_all.columns.tolist()}')

In [ ]:
# ── Spatial summary plot ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, year in enumerate(ALL_YEARS):
    ax = axes[i]
    df = firms_data[year]
    sc = ax.scatter(
        df['longitude'], df['latitude'],
        c=df['frp'], cmap='hot_r',
        s=0.3, alpha=0.5, vmin=0, vmax=200
    )
    ax.set_xlim(-125, -109)
    ax.set_ylim(32, 49)
    ax.set_title(f'{year}  —  {len(df):,} detections', fontsize=11)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.colorbar(sc, ax=ax, label='FRP (MW)')

plt.suptitle('FIRMS VIIRS S-NPP Fire Detections — Western CONUS', fontsize=14, y=1.01)
plt.tight_layout()

fig_path = f'{FIGURES_DIR}/firms_spatial_summary.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')

---
## Section 5 — Search and stream VNP14IMG via earthaccess

**VNP14IMG** is the VIIRS 375 m active fire product from Suomi NPP. It is derived from the I4 (3.74 µm MWIR) and I5 (11.45 µm LWIR) bands — the FireSat-analog channels. Collection 002 is the current version.

### Cloud streaming architecture

```
earthaccess.search_data()  →  find granule URLs at NASA Earthdata Cloud (LP DAAC)
earthaccess.open()         →  open files as streaming file objects (no download)
xr.open_dataset()          →  read variables directly into memory
extract_granule_fp()       →  parse into DataFrame, filter to domain, compute features
```

### Key variables extracted from each VNP14IMG granule

| Variable | Renamed | Description |
|---|---|---|
| `FP_latitude` | `latitude` | Fire pixel latitude |
| `FP_longitude` | `longitude` | Fire pixel longitude |
| `FP_T4` | `BT_I4` | I4 brightness temperature — MWIR (K) |
| `FP_T5` | `BT_I5` | I5 brightness temperature — LWIR (K) |
| `FP_MeanT4` | `BT_I4_bg` | Background mean BT I4 (K) |
| `FP_MeanT5` | `BT_I5_bg` | Background mean BT I5 (K) |
| `FP_MAD_T4` | `MAD_I4` | Mean absolute deviation I4 |
| `FP_MAD_T5` | `MAD_I5` | Mean absolute deviation I5 |
| `FP_power` | `frp_mw` | Fire radiative power (MW) |
| `FP_confidence` | `confidence` | Detection confidence (8=nominal, 9=high) |
| `FP_SolZenAng` | `sol_zen` | Solar zenith angle (degrees) |
| `FP_ViewZenAng` | `view_zen` | View zenith angle — needed for Beer-Lambert correction |
| `FP_MeanDT` | `BT_diff_bg` | Mean background delta-T |
| `FP_day` | `is_day` | Day (1) or night (0) flag |

### Derived features computed inline

| Feature | Formula | Physical meaning |
|---|---|---|
| `BTD` | `BT_I4 − BT_I5` | Brightness temperature difference — primary fire spectral signal |
| `BT_I4_anom` | `BT_I4 − BT_I4_bg` | Fire vs background anomaly in MWIR |
| `BTD_anom` | `BTD − (BT_I4_bg − BT_I5_bg)` | BTD anomaly relative to background context |

In [ ]:
# ── Variable definitions ───────────────────────────────────────────────────────
FP_VARS = [
    'FP_latitude', 'FP_longitude', 'FP_T4', 'FP_T5',
    'FP_MeanT4', 'FP_MeanT5', 'FP_MAD_T4', 'FP_MAD_T5',
    'FP_power', 'FP_confidence', 'FP_SolZenAng', 'FP_ViewZenAng',
    'FP_MeanDT', 'FP_day'
]

RENAME_MAP = {
    'FP_latitude'  : 'latitude',
    'FP_longitude' : 'longitude',
    'FP_T4'        : 'BT_I4',
    'FP_T5'        : 'BT_I5',
    'FP_MeanT4'    : 'BT_I4_bg',
    'FP_MeanT5'    : 'BT_I5_bg',
    'FP_MAD_T4'    : 'MAD_I4',
    'FP_MAD_T5'    : 'MAD_I5',
    'FP_power'     : 'frp_mw',
    'FP_confidence': 'confidence',
    'FP_SolZenAng' : 'sol_zen',
    'FP_ViewZenAng': 'view_zen',
    'FP_MeanDT'    : 'BT_diff_bg',
    'FP_day'       : 'is_day',
}


def extract_granule_fp(nc_file, date_str, granule_id):
    """
    Extract fire pixel variables from one open VNP14IMG NetCDF file.

    Parameters
    ----------
    nc_file    : file-like object from earthaccess.open()
    date_str   : str   — 'YYYY-MM-DD' of the granule
    granule_id : str   — unique identifier string

    Returns
    -------
    pd.DataFrame or None if no fire pixels in domain
    """
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            ds = xr.open_dataset(nc_file, engine='h5netcdf')

        # Skip granules with no fire pixels
        if 'FP_latitude' not in ds.data_vars:
            return None
        if ds['FP_latitude'].shape[0] == 0:
            return None

        # Extract variables
        fp_dict = {}
        for var in FP_VARS:
            if var in ds.data_vars:
                fp_dict[var] = ds[var].values

        df = pd.DataFrame(fp_dict).rename(columns=RENAME_MAP)

        # Spatial filter to domain
        df = df[
            (df['longitude'] >= BBOX_DICT['lon_min']) &
            (df['longitude'] <= BBOX_DICT['lon_max']) &
            (df['latitude']  >= BBOX_DICT['lat_min']) &
            (df['latitude']  <= BBOX_DICT['lat_max'])
        ].copy()

        if len(df) == 0:
            return None

        # Compute derived spectral features
        df['BTD']        = df['BT_I4'] - df['BT_I5']
        df['BT_I4_anom'] = df['BT_I4'] - df['BT_I4_bg']
        df['BTD_anom']   = df['BTD']   - (df['BT_I4_bg'] - df['BT_I5_bg'])

        # Metadata
        df['date']       = date_str
        df['granule_id'] = granule_id

        ds.close()
        return df

    except Exception:
        return None


print('Extraction functions defined.')

In [ ]:
# ── Quick test on one day before full run ─────────────────────────────────────
# Search Sept 5 2020 — a peak fire day (4631 FIRMS detections)
results = earthaccess.search_data(
    short_name   = VIIRS_SHORT_NAME,
    bounding_box = BBOX_TUPLE,
    temporal     = ('2020-09-05', '2020-09-05'),
)
print(f'Granules found for 2020-09-05: {len(results)}')

# Open and inspect one granule
files = earthaccess.open(results[:1])
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    ds_test = xr.open_dataset(files[0], engine='h5netcdf')

print(f'Variables in granule: {list(ds_test.data_vars)[:8]}...')
print(f'Fire pixels in granule: {ds_test["FP_latitude"].shape[0]}')

# Extract to DataFrame
df_test = extract_granule_fp(files[0], '2020-09-05', 'test_00')
print(f'\nFire pixels in domain: {len(df_test)}')
print(df_test[['BT_I4', 'BT_I5', 'BTD', 'frp_mw', 'confidence']].describe().round(2))
ds_test.close()

---
## Section 6 — Full multi-year VIIRS extraction

**Strategy:** For each year, identify days with ≥10 FIRMS detections (active fire days), search for VNP14IMG granules on those days, stream each granule, extract fire pixels, and save the year as a parquet file to Drive.

**Expected runtime:** 15–20 minutes per year, 90–120 minutes total for all 6 years.

**Resume safety:** If Colab disconnects mid-run, already-completed years are saved to Drive. Rerun the cell — it will skip completed years and continue from where it stopped.

**The `0%` progress bars** that appear mid-run are earthaccess briefly retrying S3 connections. They recover automatically — do not interrupt the cell.

In [ ]:
def process_year(year, firms_all, min_count=10):
    """
    Process all active fire days for one year:
      1. Identify days with >= min_count FIRMS detections
      2. For each day, search for VNP14IMG granules
      3. Stream each granule and extract fire pixel variables
      4. Save combined DataFrame to Drive as parquet

    Parameters
    ----------
    year      : int          — year to process
    firms_all : pd.DataFrame — combined FIRMS DataFrame (all years)
    min_count : int          — minimum FIRMS detections to trigger VIIRS search

    Returns
    -------
    pd.DataFrame or None
    """
    out_path = f'{VIIRS_DIR}/viirs_fp_{year}.parquet'
    if os.path.exists(out_path):
        print(f'  {year} already processed — loading from Drive')
        return pd.read_parquet(out_path)

    # Identify active fire days
    firms_year = firms_all[firms_all['acq_date'].dt.year == year]
    fire_days  = (
        firms_year.groupby('acq_date').size()
        .reset_index(name='count')
        .query(f'count >= {min_count}')
        .sort_values('acq_date')
    )
    print(f'  {year}: {len(fire_days)} active fire days to process')

    year_dfs = []

    for _, row in fire_days.iterrows():
        date_str = row['acq_date'].strftime('%Y-%m-%d')
        try:
            results = earthaccess.search_data(
                short_name   = VIIRS_SHORT_NAME,
                bounding_box = BBOX_TUPLE,
                temporal     = (date_str, date_str),
            )
            if not results:
                continue

            files = earthaccess.open(results)   # streams from NASA cloud — no download
            for i, f in enumerate(files):
                df = extract_granule_fp(f, date_str, f'{date_str}_{i:02d}')
                if df is not None and len(df) > 0:
                    year_dfs.append(df)

            time.sleep(0.3)   # brief pause between days

        except Exception as e:
            print(f'    Warning: {date_str} failed: {e}')
            continue

    if not year_dfs:
        print(f'  No data extracted for {year}')
        return None

    df_year = pd.concat(year_dfs, ignore_index=True)
    df_year.to_parquet(out_path, index=False)
    print(f'  {year}: {len(df_year):,} fire pixels saved → Drive')
    return df_year


print('process_year() defined.')

In [ ]:
# ── Run full multi-year extraction ────────────────────────────────────────────
all_year_dfs = {}

for year in ALL_YEARS:
    print(f'\n{"="*45}')
    print(f'  Processing {year}')
    print(f'{"="*45}')
    df = process_year(year, firms_all, min_count=MIN_FIRMS_COUNT)
    if df is not None:
        all_year_dfs[year] = df

# Final summary
print('\n' + '='*45)
print('  VIIRS Extraction — Complete')
print('='*45)
total = 0
for year, df in all_year_dfs.items():
    print(f'  {year}: {len(df):>9,} fire pixels')
    total += len(df)
print(f'  {"TOTAL":}: {total:>9,} fire pixels')
print('='*45)

---
## Section 7 — Verification and summary

Verify the extracted data is physically reasonable before proceeding to Module 1b.

In [ ]:
# ── Load all processed VIIRS fire pixels ─────────────────────────────────────
viirs_all = pd.concat(
    [pd.read_parquet(f'{VIIRS_DIR}/viirs_fp_{y}.parquet') for y in ALL_YEARS
     if os.path.exists(f'{VIIRS_DIR}/viirs_fp_{y}.parquet')],
    ignore_index=True
)

print('=== VIIRS Fire Pixel Dataset ===')
print(f'Total fire pixels : {len(viirs_all):,}')
print(f'Columns           : {viirs_all.columns.tolist()}')
print(f'\nKey statistics:')
print(viirs_all[['BT_I4', 'BT_I5', 'BTD', 'frp_mw', 'confidence']].describe().round(2))

print(f'\nPhysical checks:')
print(f'  BT_I4 range    : {viirs_all["BT_I4"].min():.1f} – {viirs_all["BT_I4"].max():.1f} K  (fire range ~300–500 K)')
print(f'  BTD > 0 pixels : {(viirs_all["BTD"] > 0).sum():,} / {len(viirs_all):,}  (should be majority)')
print(f'  High FRP (>100 MW): {(viirs_all["frp_mw"] > 100).sum():,}  (extreme fire events)')
print(f'  Max FRP        : {viirs_all["frp_mw"].max():.1f} MW')

print(f'\nPixels by year:')
viirs_all['year'] = pd.to_datetime(viirs_all['date']).dt.year
print(viirs_all.groupby('year').size().rename('fire_pixels').to_string())

print('\n=== Module 1a Complete ===')
print('Next: 01b_download_era5_aod.ipynb')
print('  → ERA5 atmospheric profiles co-located with fire pixels')
print('  → MODIS AOD co-located with fire pixels')

---
## Troubleshooting

| Issue | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: google.colab` | Running in local Jupyter, not Colab | Open notebook at colab.research.google.com |
| `MessageError: credential propagation unsuccessful` | Not signed into Google in browser | Sign in to Google account, then Runtime → Disconnect and delete runtime |
| `earthaccess.login()` fails | Wrong Earthdata credentials | Verify at urs.earthdata.nasa.gov |
| FIRMS returns `400 Invalid day range` | Using day range > 5 | Already fixed in `fetch_firms_year()` — uses 5-day chunks |
| FIRMS returns `404` | Wrong source name | Confirm `FIRMS_SOURCE = 'VIIRS_SNPP_SP'` for historical data |
| `0%` progress bars during VIIRS extraction | earthaccess S3 retry | Normal — waits and recovers automatically |
| Colab disconnects mid-run | Session timeout | Rerun cell — completed years are saved to Drive and will be skipped |
| `UserWarning: phony_dims` | xarray/h5netcdf version difference | Harmless warning — suppressed with `warnings.catch_warnings()` |
| Very low fire pixel count for a year | `MIN_FIRMS_COUNT` too high | Lower to 5 in config cell and rerun |

## What the output variables mean

Understanding these variables is essential before Module 2 feature engineering:

**`BT_I4` (brightness temperature, I4 band, MWIR 3.74 µm):** The primary fire detection channel. Active fires emit strongly in MWIR due to their high temperatures (>600 K). Background vegetation and soil are typically 280–320 K. A BT_I4 above ~330 K in fire-season conditions is a strong fire signal.

**`BT_I5` (brightness temperature, I5 band, LWIR 11.45 µm):** The thermal infrared reference channel. Less sensitive to fire but captures surface and atmospheric thermal emission. Used in combination with I4 for BTD calculation.

**`BTD` (brightness temperature difference, I4−I5):** The single most discriminative fire feature. Fire pixels show large positive BTD (20–100 K). Non-fire thermal emitters have smaller BTD. This is the spectral signature FireSat is designed to exploit.

**`frp_mw` (fire radiative power in MW):** An estimate of the rate of radiant energy released by the fire. Ranges from <1 MW (small smoldering fire) to >2000 MW (extreme crown fire). This is the dynamic range challenge for FireSat — the model must handle a 3-order-of-magnitude spread.

**`view_zen` (view zenith angle):** The angle between the satellite line-of-sight and vertical. Larger angles mean longer atmospheric path lengths, which affects the Beer-Lambert transmittance correction in the physics-informed loss function.